Install Dependencies

In [1]:
!pip install -q \
    sentence-transformers \
    faiss-cpu \
    pypdf \
    transformers \
    accelerate \
    bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.2/494.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.2/331.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 23.5 MB/s eta 0:00:00


Upload PDFs

In [2]:
from google.colab import files

uploaded = files.upload()

Saving amandersson,+17453674.1998.11744790.pdf to amandersson,+17453674.1998.11744790.pdf
Saving Nonspecific_Low_Back_Pain.pdf to Nonspecific_Low_Back_Pain.pdf
Saving The_Epidemiology_of_low_back_pain.pdf to The_Epidemiology_of_low_back_pain.pdf


Read PDFs

In [3]:
from pypdf import PdfReader

documents = []

for filename in uploaded.keys():
    reader = PdfReader(filename)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    documents.append({
        "source": filename,
        "text": text
    })

print(f"Loaded {len(documents)} documents")
print(documents[0]["text"][:800])

Loaded 3 documents
28 Acta Orthop Scand (Suppl 281) 1998; 69 
Epidemiology of low back pain 
Gunnar B J ANDERSSON 
Department of Orthopedic Surgery, Rush-Presbyterian-St. Luke's Medical Center, Chicago, Illinois, U.S.A. 
Spinal disorders 
Low back pain is an enormous and important clinical 
and public health problem. Although admittedly com­
mon, its epidemiology is difficult to determine pre­
cisely. There is a problem of definition and classifica­
tion. For example, the results of a survey will depend 
on definitions of the severity of pain, period of pain, 
medical attention required, disability caused by, etc. 
Further, there is a measurement problem since back 
pain can exist without objective evidence and pain 
cannot be directly measured. In addition, poor recall, 
influence of legal, social, psycholo


Remove Reference Sections

In [4]:
def remove_references(text):
    stop_words = [
        "\nreferences\n",
        "\nbibliography\n",
        "\nreference list\n",
        "\nliterature\n",
    ]
    lower = text.lower()
    for sw in stop_words:
        if sw in lower:
            return text[:lower.index(sw)]
    return text


clean_documents = []

for doc in documents:
    clean_documents.append({
        "source": doc["source"],
        "text": remove_references(doc["text"])
    })

print("Reference sections removed.")

Reference sections removed.


Paragraph Filtering

In [5]:
# ======  PARAGRAPH PROCESSING WITH FALLBACK ======

def safe_process_documents(clean_documents):
    processed_docs = []

    for doc in clean_documents:
        paragraphs = doc["text"].split("\n\n")

        good_paragraphs = []
        for p in paragraphs:
            if len(p.strip()) > 150:
                good_paragraphs.append(p)

        # Fallback if nothing survives
        if len(good_paragraphs) == 0:
            good_text = doc["text"]
        else:
            good_text = "\n\n".join(good_paragraphs)

        processed_docs.append({
            "source": doc["source"],
            "text": good_text
        })

    return processed_docs


processed_docs = safe_process_documents(clean_documents)

print("Documents processed safely.")
print(processed_docs[0]["text"][:800])

Documents processed safely.
28 Acta Orthop Scand (Suppl 281) 1998; 69 
Epidemiology of low back pain 
Gunnar B J ANDERSSON 
Department of Orthopedic Surgery, Rush-Presbyterian-St. Luke's Medical Center, Chicago, Illinois, U.S.A. 
Spinal disorders 
Low back pain is an enormous and important clinical 
and public health problem. Although admittedly com­
mon, its epidemiology is difficult to determine pre­
cisely. There is a problem of definition and classifica­
tion. For example, the results of a survey will depend 
on definitions of the severity of pain, period of pain, 
medical attention required, disability caused by, etc. 
Further, there is a measurement problem since back 
pain can exist without objective evidence and pain 
cannot be directly measured. In addition, poor recall, 
influence of legal, social, psycholo


Chunking

In [6]:
# ====== SAFE CHUNKING ======

def chunk_text(text, chunk_size=600, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if len(chunk) > 100:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


chunks = []

for doc in processed_docs:
    for ch in chunk_text(doc["text"]):
        chunks.append({
            "source": doc["source"],
            "text": ch
        })

print("Total chunks:", len(chunks))
print("\nSample chunk:\n")
print(chunks[0]["text"][:600])

Total chunks: 186

Sample chunk:

28 Acta Orthop Scand (Suppl 281) 1998; 69 
Epidemiology of low back pain 
Gunnar B J ANDERSSON 
Department of Orthopedic Surgery, Rush-Presbyterian-St. Luke's Medical Center, Chicago, Illinois, U.S.A. 
Spinal disorders 
Low back pain is an enormous and important clinical 
and public health problem. Although admittedly com­
mon, its epidemiology is difficult to determine pre­
cisely. There is a problem of definition and classifica­
tion. For example, the results of a survey will depend 
on definitions of the severity of pain, period of pain, 
medical attention required, disability caused by, et


Embeddings

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embedding shape: (186, 384)


FAISS Index

In [8]:
import faiss

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

final_chunks = chunks  # canonical chunk list

print("FAISS index size:", index.ntotal)

FAISS index size: 186


Retriever Function

In [9]:
def retrieve_context(query, k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )
    distances, indices = index.search(query_embedding, k)

    results = []
    for idx in indices[0]:
        if idx < len(final_chunks):
            results.append(final_chunks[idx])

    return results

Load Local LLM

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded.")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LLM loaded.


Doctor Prompt

In [11]:
def doctor_prompt(context, question):
    return f"""
You are a licensed medical doctor assistant.

Rules:
- Do NOT diagnose diseases
- Do NOT prescribe medications
- Provide general medical advice only
- Use calm, professional language
- Mention warning signs (red flags) when relevant

Medical context (may be incomplete):
{context}

Patient question:
{question}

Doctor-style response:
"""

RAG Chat Function

In [12]:
def rag_chat(question, k=3, max_tokens=300):
    retrieved = retrieve_context(question, k)
    context = "\n\n".join([r["text"] for r in retrieved])

    prompt = doctor_prompt(context, question)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.4,
        top_p=0.9
    )

    response = tokenizer.decode(output[0], skip_special_tokens=True)
    return response.split("Doctor-style response:")[-1].strip()

Prompt Test in English

In [13]:
print(
    rag_chat(
        "My lower back hurts. Should I rest in bed or keep moving?"
    )
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Based on the available evidence, it appears that staying active and continuing with your normal daily activities, as much as possible, may be more beneficial for managing lower back pain than bed rest. However, it's important to note that everyone's situation is unique, and individual responses to treatment can vary greatly. If your pain is severe or accompanied by other concerning symptoms such as fever, difficulty walking, or numbness or tingling in your legs, it would be best to consult with a healthcare professional for further evaluation. Additionally, if you find that staying active worsens your pain, it may be necessary to modify your activities or seek additional interventions, such as physical therapy or education about proper body mechanics.


Prompt Test in French

In [14]:
'''print(
    rag_chat("J’ai mal au bas du dos. Est-ce que je dois rester au lit ?")
)'''

'print(\n    rag_chat("J’ai mal au bas du dos. Est-ce que je dois rester au lit ?")\n)'

Install Gradio

In [15]:
# ====== Install Gradio for chatbot UI ======
!pip install -q gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 704.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 11.1 MB/s eta 0:00:00


Gradio Chatbot UI

In [16]:
import gradio as gr

def chat_fn(message, history):
    try:
        response = rag_chat(message)
        return response
    except Exception as e:
        return f"⚠️ Error: {str(e)}"


with gr.Blocks(title="Back Pain Medical Chatbot") as demo:
    gr.Markdown(
        """
        # 🧑‍⚕️ Back Pain Medical Chatbot
        This chatbot provides **general medical information** about back pain.
        It does **not** replace a healthcare professional.
        """
    )

    chatbot = gr.ChatInterface(
        fn=chat_fn,
        chatbot=gr.Chatbot(height=400),
        textbox=gr.Textbox(
            placeholder="Ask a question about back pain (English or French)...",
            scale=7
        ),
        title="Doctor-Style RAG Chatbot",
        description="Powered by local RAG (FAISS + LLM). No external APIs."
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://39cc65ac089788addd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
